# Test: Mistral-Small-3.2-24B-Instruct-2506 — Validator V2

GPU 4 (shared with Qwen3-32B, 45% utilization), Port 8003

**Prerequisites:** vLLM server running on port 8003 with `--tokenizer-mode mistral`

In [1]:
import sys
sys.path.insert(0, '/storage/data/AgenticCyOps_Private')

from models.utils import MistralSmall

model = MistralSmall()
print('Model config:')
model.get_config()

/home/student/.conda/envs/agenticcyops/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model config:


{'model_id': 'mistralai/Mistral-Small-3.2-24B-Instruct-2506',
 'model_path': '/storage/data/AgenticCyOps_Private/models/mistralai/Mistral-Small-3.2-24B-Instruct-2506',
 'role': 'validator_2',
 'architecture': 'dense',
 'total_params': '24B',
 'gpu_assignment': '4',
 'gpu_shared': True,
 'port': 8003,
 'base_url': 'http://localhost:8003/v1',
 'tool_call_parser': 'mistral',
 'tokenizer_mode': 'mistral',
 'context_length': 128000}

## 1. Health Check

In [2]:
assert model.health_check(), 'Server not running on port 8003!'
print('Health check passed')

Health check passed


## 2. List Models

In [3]:
models = model.list_models()
for m in models:
    print(f'  {m.id}')

  /storage/data/AgenticCyOps_Private/models/mistralai/Mistral-Small-3.2-24B-Instruct-2506


## 3. Chat Completions

In [4]:
messages = [
    {'role': 'system', 'content': 'You are a SOC analyst. Be concise.'},
    {'role': 'user', 'content': 'What is a lateral movement attack? One sentence.'}
]

resp = model.chat(messages, max_tokens=100)
print('Basic chat:', resp.choices[0].message.content)

Basic chat: A lateral movement attack is when an attacker, already inside a network, moves from one system or account to another to expand access and evade detection.


In [5]:
# Deterministic
resp = model.chat_deterministic(messages, max_tokens=100)
print('Deterministic:', resp.choices[0].message.content)

Deterministic: A lateral movement attack is when an attacker, already inside a network, moves through it to access other systems or data.


In [6]:
# Creative
resp = model.chat_creative(messages, max_tokens=100)
print('Creative:', resp.choices[0].message.content)

Creative: A lateral movement attack is when an intruder, already inside a network, moves attentively between systems to find and access valuable data.


In [7]:
# Streaming
stream = model.chat(messages, max_tokens=100, stream=True)
print('Streaming: ', end='')
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print()

Streaming: A lateral movement attack is when an attacker, already inside a network, moves through it to access other systems or data.


## 4. Validate (Primary Use Case)

In [8]:
# Safe proposal
resp = model.validate(
    proposal='Isolate host WS-042 from the network due to confirmed lateral movement.',
    context='Alert: Lateral movement detected from WS-042 to DC-01 via PsExec. Source IP 10.0.5.42.'
)
print('Safe proposal:')
print(resp.choices[0].message.content)

Safe proposal:
{
  "approved": true,
  "confidence": 0.95,
  "reasoning": "The proposed action is appropriate given the incident context. Lateral movement detected via PsExec is a strong indicator of a potential compromise. Isolating the host WS-042 will prevent further movement within the network and limit the potential damage.",
  "risks": [
    "Isolating WS-042 may disrupt ongoing legitimate processes or services that rely on this host.",
    "There is a risk that the attacker may have already moved to other systems before isolation.",
    "Isolation may alert the attacker, causing them to take evasive actions."
  ]
}


In [9]:
# Dangerous proposal
resp = model.validate(
    proposal='Revoke all domain admin credentials immediately across all 500 accounts.',
    context='Alert: Single phishing email detected. No evidence of credential compromise.'
)
print('Dangerous proposal:')
print(resp.choices[0].message.content)

Dangerous proposal:
{
  "approved": false,
  "confidence": 0.95,
  "reasoning": "The proposed action is disproportionate to the incident context. A single phishing email with no evidence of credential compromise does not warrant revoking all domain admin credentials. This action would cause significant disruption and downtime for legitimate users and administrators.",
  "risks": [
    "Unnecessary disruption of business operations",
    "Loss of productivity due to credential revocation",
    "Potential for human error during mass credential revocation",
    "Risk of further incidents due to rushed or improper credential reissuance"
  ]
}


In [10]:
# Batch validate
proposals = [
    {'proposal': 'Block IP 10.0.5.12 at firewall.', 'context': 'Confirmed C2 from 10.0.5.12.'},
    {'proposal': 'Delete all firewall rules.', 'context': 'Minor config drift detected.'},
]
results = model.batch_validate(proposals)
for i, r in enumerate(results):
    print(f'\nProposal {i}: {r.choices[0].message.content[:120]}...')


Proposal 0: {
  "approved": true,
  "confidence": 0.95,
  "reasoning": "Blocking the confirmed C2 IP address at the firewall is a st...

Proposal 1: {
  "approved": false,
  "confidence": 0.95,
  "reasoning": "Deleting all firewall rules is an extreme action that would...


## 5. Tool Calling

In [11]:
tools = [{
    'type': 'function',
    'function': {
        'name': 'query_siem',
        'description': 'Search SIEM logs',
        'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']}
    }
}]
tc_messages = [{'role': 'user', 'content': 'Search SIEM for failed logins from 10.0.5.12'}]

try:
    resp = model.tool_call(tc_messages, tools)
    tc = resp.choices[0].message.tool_calls
    if tc:
        print(f'Tool call: {tc[0].function.name}({tc[0].function.arguments})')
    else:
        print('No tool call (expected if served without --enable-auto-tool-choice)')
except Exception as e:
    print(f'Tool calling not available (expected for default validator config): {e}')

No tool call (expected if served without --enable-auto-tool-choice)


## 6. Structured Output

In [12]:
json_messages = [
    {'role': 'system', 'content': 'Respond with JSON only.'},
    {'role': 'user', 'content': 'Classify: "Failed SSH from 10.0.5.12". Return {"severity": str, "confidence": float}'}
]
resp = model.chat_json(json_messages, temperature=0.0, max_tokens=200)
print('JSON mode:', resp.choices[0].message.content)

JSON mode: {
  "severity": "medium",
  "confidence": 0.8
}


## 7. Batch Chat

In [13]:
batches = [
    [{'role': 'user', 'content': 'What is phishing? One sentence.'}],
    [{'role': 'user', 'content': 'What is ransomware? One sentence.'}],
]
results = model.batch_chat(batches, max_tokens=80)
for i, r in enumerate(results):
    print(f'Batch {i}: {r.choices[0].message.content}')

Batch 0: Phishing is a fraudulent practice where attackers impersonate legitimate entities to trick individuals into revealing sensitive information, such as passwords or credit card details.
Batch 1: Ransomware is a type of malicious software that encrypts a victim's files and demands payment, usually in cryptocurrency, in exchange for their restoration.


## 8. Token Usage

In [14]:
resp = model.chat(messages, max_tokens=100)
usage = resp.usage
print(f'Prompt tokens:     {usage.prompt_tokens}')
print(f'Completion tokens: {usage.completion_tokens}')
print(f'Total tokens:      {usage.total_tokens}')

Prompt tokens:     24
Completion tokens: 25
Total tokens:      49


## 9. Get Config

In [15]:
import json
print(json.dumps(model.get_config(), indent=2))

{
  "model_id": "mistralai/Mistral-Small-3.2-24B-Instruct-2506",
  "model_path": "/storage/data/AgenticCyOps_Private/models/mistralai/Mistral-Small-3.2-24B-Instruct-2506",
  "role": "validator_2",
  "architecture": "dense",
  "total_params": "24B",
  "gpu_assignment": "4",
  "gpu_shared": true,
  "port": 8003,
  "base_url": "http://localhost:8003/v1",
  "tool_call_parser": "mistral",
  "tokenizer_mode": "mistral",
  "context_length": 128000
}


## Summary

All tests passed if no cells raised exceptions above.